# System Recommenders - Final Project 2025

### 🎯 Objective

Develop a recommender system that suggests short videos to users based on user preferences, interaction histories, and video content using the KuaiRec dataset. 

The challenge is to create a personalised and scalable recommendation engine similar to those used in platforms like TikTok or Kuaishou.

### 📥 Imports

In [2]:
import pandas as pd
import numpy as np

from scipy.sparse import csr_matrix
from implicit.als import AlternatingLeastSquares
from langdetect import detect
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import BertTokenizer

### 📊 Download Dataset

We will use the **KuaiRec dataset**, a large-scale, fully-observed dataset collected from the Kuaishou short-video platform.

It contains:

- **User interactions** (views, likes, etc.)
- **Video metadata** (video ID, tags, etc.)
- **Timestamps**

More info: [KuaiRec Paper](https://arxiv.org/abs/2202.10842)

**Download dataset**

1. <ins>First option : Downloading Dataset via wget<ins>

In [3]:
%%bash
if [ ! -d "./data_final_project" ]; then
  wget --no-check-certificate 'https://drive.usercontent.google.com/download?id=1qe5hOSBxzIuxBb1G_Ih5X-O65QElollE&export=download&confirm=t&uuid=b2002093-cc6e-4bd5-be47-9603f0b33470' -O KuaiRec.zip
  unzip KuaiRec.zip -d ../data_final_project
else
  echo "Directory './data_final_project' already exists. Skipping download."
fi

Directory './data_final_project' already exists. Skipping download.


2. <ins>Second option : Downloading dataset via Google Drive<ins>

If the data is not downloaded by the wget because of a Connection Refused you might download it via this [link](https://drive.google.com/file/d/1qe5hOSBxzIuxBb1G_Ih5X-O65QElollE/view) and place it at the project root.

In [4]:
""" Uncomment if you need to (and if the first option did not work correctly)
%%bash
unzip KuaiRec.zip -d data_final_project
mv "data_final_project/KuaiRec 2.0/" data_final_project/KuaiRec
"""

' Uncomment if you need to (and if the first option did not work correctly)\n%%bash\nunzip KuaiRec.zip -d data_final_project\nmv "data_final_project/KuaiRec 2.0/" data_final_project/KuaiRec\n'

From this dataset we obtain the following files :

```bash
KuaiRec
  ├── data
  │   ├── big_matrix.csv          
  │   ├── small_matrix.csv
  │   ├── social_network.csv
  │   ├── user_features.csv
  │   ├── item_daily_features.csv
  │   └── item_categories.csv
  │   └── kuairec_caption_category.csv
```

- `interactions_train.csv`: historical user-item interactions for training.
- `interactions_test.csv`: user-item pairs to score during testing.
- `sample_submission.csv`: a template showing the expected output format.
- `video_metadata.csv`: metadata including tags or content-related features.

![image](img/KuaiRec.png)

## **1️⃣ Dataset Preprocessing**
📝 Associated Tasks :
- Load and inspect the dataset.
- Handle missing or inconsistent data.
- Merge metadata for content-based models if necessary.

---


<div style="background-color:#cccccc; padding: 10px; border-radius: 5px; border: 1px solid #aaa; font-weight: bold; color:#111;">
    💡 For more details about the different analyses and pre-processing decisions made on the available datasets, <a href="./EDA/EDA.ipynb" style="color:#1a73e8;">feel free to click here and go to the EDA notebook</a>.
</div>


---

However, here is a small recap of the observations made : 

- **User Interactions**: 
  - The majority of users have **fewer than 3,000 interactions**.
  - There are a few **outliers** with interactions exceeding 6,000.

- **Item Popularity**:
  - There are very few items that receive **high levels of interaction**.
  - Many items, while not extremely popular, still receive a fair amount of attention.
  - A small subset of items are truly **highly popular**.

- **Time-Based Trends**:
  - However, interactions peak towards the **end of the week**, specifically from **Friday to Sunday**.
  - **Late-night hours (0:00 - 3:00 a.m.)** see the highest levels of activity.
  - **From 10:00 to 20:00** we see less activity
  - The time of day or day of the week has **minimal to no impact** on the watch ratio.

- **User Behavior**:
  - The **top 10 most active users** show a distinct dip in activity around **12:00 p.m.** and **4:00 p.m.**.

- **Video Length**:
  - **Shorter videos** (up to 30 seconds) receive **more interactions** and have a **higher watch ratio**.
  - Conversely, **longer videos** tend to have **fewer interactions** and a lower watch ratio.
  - Videos that are **no longer than 30 seconds** appear to have the optimal watch ratio.

- **Watch Ratio**:
  - **Half of the videos have less then 75% watch ratio.**
  - Only **0.1% of the videos have more than 18** of watch ratio. These are extreme outliers, we can either drop them or try and normalize them.

- **Video Type:**

    - **"Ad" videos have significantly lower interaction** rates compared to Normal videos, which are more engaging for users.
    - Users tend to engage less with promotional or advertisement-type content.

- **User Preferences:**

    - There is a clear preference for **Short Imports videos**. These videos are viewed and interacted with more often compared to longer or different upload types.

- **Video Format:**

    - The **1280x720 resolution is the optimal format** for maximizing user interaction and watch ratio. Other formats either fall short or show no significant improvement.

- **Video Privacy Settings:**

    - As expected, **public videos receive more interactions** than private or only friends videos. This could be due to the higher volume of public content being uploaded, which in turn increases the overall interactions.

    - Private or only friends videos tend to have a smaller, more targeted audience, reducing their interaction rates.

- **Video Age**:
    - **Recent videos (lower video age in days) tend to have more interactions**



### Load Datasets

Here as we have shown in the [EDA notebook](./EDA/EDA.ipynb) with more details :

1. We have to load the dataset we will use.

    We will be using four datasets :
    - `big_matrix.csv`
        - Contains user interactions with the videos.
        - It will serve for the training.
    - `small_matrix.csv`
        - Similar to big_matrix, but smaller in size
        - It will serve for the testing.
    - `item_daily_features.csv`
        - Contains informations about the video (e.g. number of likes, shares, reports ...).
        - It will be merged with both the small and big matrices to provide additional features.
    - `kuairec_caption_category.csv`
        - Contains informations about the videos' categories and captions.
        - It will also be merged with both the small and big matrices to enrich them with contextual metadata.

2. We perform some preprocessing steps:
    - Convert columns to more appropriate data types
    - Remove highly correlated columns
    - Cap the `watch_ratio` for videos above 2.34, as these represent the top 5% (as shown in the [EDA notebook](./EDA/EDA.ipynb)). This helps normalize the `watch_ratio`, which is an important metric.


In [5]:
interactions = pd.read_csv("./data_final_project/KuaiRec/data/big_matrix.csv")
small_interactions = pd.read_csv("./data_final_project/KuaiRec/data/small_matrix.csv")
item_features = pd.read_csv("./data_final_project/KuaiRec/data/item_daily_features.csv")
captions = pd.read_csv("./data_final_project/KuaiRec/data/kuairec_caption_category.csv", lineterminator='\n')

# Data Cleaning Cell
def clean_df(df):
    df = df.dropna()
    df = df.drop_duplicates()  
    return df  

def clean_df_timestamp(df):
    df = clean_df(df)
    df = df[df["timestamp"] >= 0]
    df = df[df["watch_ratio"] <= 200]
    return df

item_features = clean_df(item_features)
item_features = item_features.drop_duplicates(subset='video_id')
captions = clean_df(captions)
captions = captions.drop_duplicates(subset='video_id')
train_df = clean_df_timestamp(interactions)
test_df = clean_df_timestamp(small_interactions)

item_features['upload_dt'] = pd.to_datetime(item_features['upload_dt'])
item_features['date'] = pd.to_datetime(item_features['date'], format='%Y%m%d')

captions = captions[['video_id', 'caption', 'first_level_category_name', 'second_level_category_name','third_level_category_name']]
captions.fillna('UNKNOWN', inplace=True)

We drop some columns from the training and testing DataFrames that are not needed:

- `play_duration` : 
    - This column is redundant, as it provides the same information as `watch ratio`
- `date`(from the interaction data):
    - This represents the interaction date, but we use the `item_features` instead, which corresponds to the post's publication date.
- `video_duration` :
    - Not needed, as it is already included in the item_features DataFrame.
- `time` and `timestamp` :
    - These are not relevant for our model and are therefore removed.

In [6]:
to_drop = ['date', 'play_duration', 'video_duration', 'time', 'timestamp']

train_df.drop(columns=to_drop, inplace=True, errors='ignore')
test_df.drop(columns=to_drop, inplace=True, errors='ignore')

We apply a cap of 2.34 to the `watch_ratio` to prevent outliers.

This decision is mainly to avoid suggesting videos with excessively high watch ratios (some videos have a ratio as high as 600, as shown in the [EDA notebook](./EDA/EDA.ipynb)). 

It also ensures that we include videos with more moderate watch ratios in the recommendations.

In [7]:
train_df['watch_ratio'] = train_df['watch_ratio'].apply(lambda x: min(x, 2.34))
test_df['watch_ratio'] = test_df['watch_ratio'].apply(lambda x: min(x, 2.34))

In [9]:
len(item_features.columns)

58

Since the item_features DataFrame contains 58 columns, we will drop columns that have a correlation higher than 80% to reduce redundancy.

In [10]:
correlation = item_features[[
       'video_duration', 'video_width',
       'video_height', 'music_id',
       'show_cnt', 'show_user_num', 'play_cnt', 'play_user_num',
       'play_duration', 'complete_play_cnt', 'complete_play_user_num',
       'valid_play_cnt', 'valid_play_user_num', 'long_time_play_cnt',
       'long_time_play_user_num', 'short_time_play_cnt',
       'short_time_play_user_num', 'play_progress', 'comment_stay_duration',
       'like_cnt', 'like_user_num', 'click_like_cnt', 'double_click_cnt',
       'cancel_like_cnt', 'cancel_like_user_num', 'comment_cnt',
       'comment_user_num', 'direct_comment_cnt', 'reply_comment_cnt',
       'delete_comment_cnt', 'delete_comment_user_num', 'comment_like_cnt',
       'comment_like_user_num', 'follow_cnt', 'follow_user_num',
       'cancel_follow_cnt', 'cancel_follow_user_num', 'share_cnt',
       'share_user_num', 'download_cnt', 'download_user_num', 'report_cnt',
       'report_user_num', 'reduce_similar_cnt', 'reduce_similar_user_num',
       'collect_cnt', 'collect_user_num', 'cancel_collect_cnt',
       'cancel_collect_user_num']].corr()

upper = correlation.where(np.triu(np.ones(correlation.shape), k=1).astype(bool))

to_drop = [column for column in upper.columns if any(upper[column] > 0.8)]
# These columns are dropped as we don't need them anymore 
# (merged in previous cell or just not needed for collaborative-filtering)
item_features.drop(columns=to_drop, inplace=True, errors='ignore')


## **2️⃣ Feature Engineering**
📝 Associated Tasks :
- Create meaningful features from interaction and metadata (e.g., content tags, user activity history).
- Build user-item interaction matrix.
- Optionally extract time-based or popularity-based features.

---

### Basic additional features

As suggested, we will now create some new features for the datasets. The first two are:

- `video_age`: the number of days since the video was posted
- `is_short_video`: a boolean indicating whether the video is shorter than 30 seconds

As shown in the [EDA notebook](./EDA/EDA.ipynb), shorter and more recent videos tend to receive more interactions than other types of videos.

In [11]:
item_features['video_age'] = (item_features['date'] - item_features['upload_dt']).dt.days
item_features['is_short_video'] = (item_features['video_duration'].fillna(0) <= 30).astype(int)

We remove the following columns as we don't need them in the future.

- `music_id`,  `video_tag_id`, `video_tag_name` : These do not provide additional useful information for our model.
- `play_progress`, `play_duration` : These are either redundant or already captured by the `watch_ratio`.
- `video_duration`, `date`, `upload_dt` : These were used to create derived features (`video_age` and `is_short_video`) so they are no longer necessary.
- `time` : We already have sufficient temporal information from other features.

In [12]:
to_drop =   [
                'date', 'upload_dt', 'video_duration', 'music_id',
                'video_tag_name', 'play_progress', 'video_tag_id',
                'time', 'play_duration'
            ]
item_features.drop(columns=to_drop, inplace=True, errors='ignore')

Let's now merge the item_features dataset, containing some insights on the videos such as number of likes, to our train_df and test_df.

In [13]:
train_df = pd.merge(train_df, item_features, on='video_id', how='left')
test_df = pd.merge(test_df, item_features, on='video_id', how='left')

Here we will take care of the language of the videos, by finding the different languages for each videos as shown in the [EDA notebook](./EDA/EDA.ipynb). 

In [14]:
def detect_language(text):
    try:
        return detect(text)
    except:
        return "UNKNOWN"
    
captions['language'] = captions['caption'].apply(detect_language)

# Load the BERT tokenizers
tokenizer_cn = BertTokenizer.from_pretrained("bert-base-chinese")
tokenizer_kr = BertTokenizer.from_pretrained("beomi/kcbert-base")

# Tokenize the text based on language
def tokenize_by_lang(text, lang):
    text = str(text)
    if lang == 'zh-cn':
        return ' '.join(tokenizer_cn.tokenize(text))
    elif lang == 'ko':
        return ' '.join(tokenizer_kr.tokenize(text))
    else:
        return 'UNKNOWN'

captions['tokenized'] = captions.apply(lambda x: tokenize_by_lang(x['caption'], x['language']), axis=1)

### Engagement Score

To effectively train our ALS (Alternating Least Squares) recommendation model, we need a **proxy rating** to represent user interest or satisfaction with a video. 

We thus can define a **custom engagement score** that combines multiple implicit feedback signals. 

Based on insights from our exploratory data analysis ([see EDA notebook](./EDA/EDA.ipynb)), we define the **engagement score** using a combination of behavioral, content-related, and video metadata features.

These features capture how likely a user is to have positively interacted with a video, making this score a suitable proxy for collaborative filtering.

#### Key features used to compute the engagement score:

- **Watch Ratio**: Proportional score based on the ratio of the video watched. A higher watch ratio directly increases the score.
- **Short Videos (`is_short_video`)**: Users show a preference for short videos, so these receive a small bonus.
- **Video Age**: Newer videos are generally more engaging; the score includes a time decay bonus favoring recent uploads.
- **Video Type**: Videos marked as **"AD"** receive a penalty.
- **Upload Type**: Some formats are known to drive higher engagement (e.g., `"ShortImport"`, `"StartCamera"`), and are weighted accordingly.
- **Visibility Status**: Public videos are more likely to be seen and engaged with, hence receive a bonus.
- **Video Resolution**: The preferred format is `1280x720`. Videos with this resolution receive more interactions.
- **User Interaction Metrics**:
  - Positive interactions (`like_cnt`, `comment_cnt`, `share_cnt`, etc.) are weighted logarithmically to contribute to the score.
  - Negative feedback (`cancel_like_cnt`, `cancel_follow_cnt`, `report_cnt`) reduces the score, also using logarithmic scaling.

Finally, the score is **min-max normalized** between `0` and `1` to ensure comparability across the dataset and improve model convergence.

This engagement score becomes our **implicit rating** that the ALS model will learn to predict.

*Note: The weights assigned to each component are based on intuition; they can be fine-tuned further through experimentation.*

In [15]:
def build_engagement_score(df):
    df["engagement_score"] = 0
    
    # Watch Ratio
    if 'watch_ratio' in df.columns:
        df["engagement_score"] += df['watch_ratio'].fillna(0) * 10
    
    # is_short_video
    df["engagement_score"] += df['is_short_video'].fillna(0) * 3
    
    # Video age
    if 'video_age' in df.columns:
        max_age = 365
        normalized_age = np.minimum(df['video_age'].fillna(max_age), max_age) / max_age
        # Newer videos get up to 2 points bonus
        df["engagement_score"] += (1 - normalized_age) * 2
    
    
    if 'video_type' in df.columns:
        df["engagement_score"] += np.where(
            df['video_type'] == 'AD',
            -3,  # penalty for ads
            2    # bonus for regular content
        )
    
    if 'visible_status' in df.columns:
        df["engagement_score"] += np.where(
            df['visible_status'] == 'public',
            2,  
            -1
        )
    
    if 'upload_type' in df.columns:
        upload_type_weights = {
            'ShortImport': 3,     # Short imported videos tend to be high quality
            'StartCamera': 2.5,
            'Knowle': 2,
            'Web': 1.5,
            'LongImport': 1,
            'UNKNOWN': 0,
            'LongCamera': 0.5,
            'PictureSet': 0.5,
            'LongPicture': 0.5,
            'ACurlVideo': 0.5,
            'followShot': 0.5,
            'ShareFromOtherApp': 0.5,
            'SameFrame': 0,
            'PictureCopy': 0,
            'FlashPhoto': 0,
            'PhotoCopy': 0,
            'LocalCollection': 0,
            'LocalInteraction': 0
        }
        df["engagement_score"] += df['upload_type'].map(upload_type_weights).fillna(0)
    
    engagement_columns = {
        'like_cnt': 0.5,
        'comment_cnt': 0.7,
        'share_cnt': 0.8,
        'collect_cnt': 0.6,
        'follow_cnt': 0.9,
        'complete_play_cnt': 0.7,
        'valid_play_cnt': 0.5,
        'reply_comment_cnt': 0.6,
        'comment_like_cnt': 0.4
    }
    
    for col, weight in engagement_columns.items():
        if col in df.columns:
            df["engagement_score"] += np.minimum(np.log1p(df[col].fillna(0)) * weight, 10)


    penalty_columns = {
        'cancel_like_cnt': 0.4,
        'cancel_follow_cnt': 0.5,
        'report_cnt': 0.7
    }
    for col, weight in penalty_columns.items():
        if col in df.columns:
            df["engagement_score"] -= np.minimum(np.log1p(df[col].fillna(0)) * weight, 10)

    if 'video_width' in df.columns:
        df["engagement_score"] += np.where(df['video_width'] >= 720, 0.5, 0)

    if 'video_height' in df.columns:
        df["engagement_score"] += np.where(df['video_height'] >= 1280, 0.5, 0)
    
    # Here we normalize the score    
    min_score = df["engagement_score"].min()
    max_score = df["engagement_score"].max()

    df["engagement_score"] = (df["engagement_score"] - min_score) / (max_score - min_score)
    
    return df

test_df = build_engagement_score(test_df)
train_df = build_engagement_score(train_df)

## **3️⃣ Model Development**
📝 Associated Tasks :
- Choose a recommendation approach:
    - Collaborative filtering (e.g., ALS, Matrix Factorisation)
    - Content-based filtering
    - Sequence-aware models
    - Hybrid approaches
- Train and validate your model on the training set.

---

### ALS Model

The `engagement_score` we made in the previous section will act as the rating of our video. 

Therefore we can use ALS, which is a collaborative filtering model, that will try to guess the engagement score. The higher it is, the more engaging the video is.

The ALS Model was choosen here as it will take the different studied metrics in the one column `engagement_score` which will make more impact.

#### User item matrix 
First we will create the user-item matrix, which is essential for the ALS algorithm, to map the different users, videos and their engagement score.

In [16]:
# Get Unique user and videos
user_ids_train = train_df['user_id'].unique()
video_ids_train = train_df['video_id'].unique()

# Compute index for each user and videos
user_to_index = {user_id: idx for idx, user_id in enumerate(user_ids_train)}
video_to_index = {video_id: idx for idx, video_id in enumerate(video_ids_train)}
index_to_user = {idx: user_id for user_id, idx in user_to_index.items()}
index_to_video = {idx: video_id for video_id, idx in video_to_index.items()}

# add the index to the train and test
train_df['user_index'] = train_df['user_id'].map(user_to_index)
train_df['video_index'] = train_df['video_id'].map(video_to_index)

test_df['user_index'] = test_df['user_id'].map(user_to_index)
test_df['video_index'] = test_df['video_id'].map(video_to_index)

row = train_df['user_index'].values
col = train_df['video_index'].values

data = train_df['engagement_score'].values

n_users = train_df['user_index'].max() + 1
n_items = train_df['video_index'].max() + 1
    
user_item_matrix = csr_matrix((data, (row, col)), shape=(n_users, n_items))

#### Model Training
Here we train our ALS Model with the `user_item_matrix` 💪

The different parameters were found with the code you can find at the end of this notebook in the  "6️⃣ Trying to improve our model" section.

In [17]:
model = AlternatingLeastSquares(
    factors=100,
    regularization=0.1,
    iterations=15,
    use_gpu=False,
    alpha=40
)

model.fit(user_item_matrix.T) 

/opt/homebrew/Caskroom/miniconda/base/lib/python3.12/site-packages/implicit/cpu/als.py:95: RuntimeWarning: OpenBLAS is configured to use 8 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()
/opt/homebrew/Caskroom/miniconda/base/lib/python3.12/site-packages/implicit/utils.py:164: ParameterWarning: Method expects CSR input, and was passed csc_matrix instead. Converting to CSR took 0.054756879806518555 seconds
  warnings.warn(


  0%|          | 0/15 [00:00<?, ?it/s]

### Content-Based Model
Here we can also add a content-based model in order to have an hybrid approach based on a collaborative filtering model (ALS), and the following content-based model.

Our model will be focusing on the language of the video, but also on the categories of the video that we can find in the `first_level_category_name`, `second_level_category_name`, `third_level_category_name`.

In [18]:
tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(captions['tokenized'])
text_sim = cosine_similarity(tfidf_matrix)

n_videos = len(captions)

first_level_sim = (captions['first_level_category_name'].values[:, None] == captions['first_level_category_name'].values).astype(float)
second_level_sim = (captions['second_level_category_name'].values[:, None] == captions['second_level_category_name'].values).astype(float)
third_level_sim = (captions['third_level_category_name'].values[:, None] == captions['third_level_category_name'].values).astype(float)

text_weight = 0.5
first_level_weight = 0.3
second_level_weight = 0.15
third_level_weight = 0.05

combined_sim = (
    text_sim * text_weight +
    first_level_weight * first_level_sim +
    second_level_weight * second_level_sim +
    third_level_weight * third_level_sim
)

combined_sim = np.clip(combined_sim, 0, 1)

indices = pd.Series(captions.index, index=captions['video_id']).drop_duplicates()

## **4️⃣ Recommendation Algorithm**
📝 Associated Tasks :
- Predict which videos are likely to be enjoyed by each user in the test set.
- Generate a top-N ranked list of recommendations for each user.

---
In this section, we will write three auxiliary functions to get the recommendations.

The first one is designed to get the recommendations from the ALS Model.

In [19]:
def get_als_recommendations(model, user_item_matrix, user_id, user_to_index, video_to_index, index_to_video, n=10):
    if user_id not in user_to_index:
        return []
    
    user_idx = user_to_index[user_id]
    # Items the user has already interacted with (adjusted for training)
    already_interacted = set(user_item_matrix[user_idx].indices)
    # U * V^T => computes the final score
    scores = model.user_factors[user_idx].dot(model.item_factors.T)
    
    item_scores = [(item_id, scores[item_id])
                  for item_id in range(len(scores))
                  if item_id not in already_interacted]
    # Sort and select top-N items
    item_scores.sort(key=lambda x: x[1], reverse=True)
    top_items = [index_to_video[item[0]] for item in item_scores[:n]]
    return top_items

The second one is to get recommendations from the hybrid model.

In [20]:
def get_content_recommendations(video_id, indices, combined_sim, captions_df, num_recommend=10):
    if video_id not in indices:
        return []
    
    idx = indices[video_id]
    
    if idx >= combined_sim.shape[0]:
        return []
    
    sim_scores = list(enumerate(combined_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    top_similar = sim_scores[1:num_recommend+1]
    video_indices = [i[0] for i in top_similar]
    
    valid_indices = [i for i in video_indices if i < len(captions_df)]
    
    if not valid_indices:
        return []
        
    return captions_df['video_id'].iloc[valid_indices].tolist()

The following function will give popular recommendations.

The purpose of this function is mostly to solve our ALS and content-based model "Cold Start" problem, meaning if the video or the user was not available in training, and our two previous algorithm cannot manage to find recommendations, we will recommend popular videos.

In [21]:
def get_popular_recommendations(train_df, num_recommend=10):
    video_counts = train_df['video_id'].value_counts().reset_index()
    video_counts.columns = ['video_id', 'count']
    return video_counts.head(num_recommend)['video_id'].tolist()

popular_recs = get_popular_recommendations(train_df, num_recommend=10)

Here is our final function, which combines all of the recommendation algorithms (from the ALS model and the content-based model), with a fallback to the most popular videos if neither can provide recommendations.

In [22]:
def get_hybrid_recommendations(user_id, video_id, model, user_item_matrix, 
                              user_to_index, video_to_index, index_to_video,
                              indices, combined_sim, captions,
                              popular_recs, n=10, weight_als=0.7):

    
    als_recs = []
    content_recs = []
    
    # Try to get ALS recommendations for the user
    if user_id in user_to_index:
        als_recs = get_als_recommendations(
            model, user_item_matrix, user_id, 
            user_to_index, video_to_index, index_to_video, n=n
        )
    
    # Try to get content-based recommendations for the video
    if video_id in indices:
        content_recs = get_content_recommendations(
            video_id, indices, combined_sim, captions, num_recommend=n
        )
    
    # If both approaches returned results, combine them with weighted scores
    if als_recs and content_recs:
        # Create a score dictionary for each recommendation type
        als_scores = {vid: (n - i) / n for i, vid in enumerate(als_recs)}
        content_scores = {vid: (n - i) / n for i, vid in enumerate(content_recs)}
        
        # Combine all unique videos
        all_videos = set(als_recs) | set(content_recs)
        
        # Calculate hybrid scores
        hybrid_scores = []
        for vid in all_videos:
            als_score = als_scores.get(vid, 0)
            content_score = content_scores.get(vid, 0)
            hybrid_score = weight_als * als_score + (1 - weight_als) * content_score
            hybrid_scores.append((vid, hybrid_score))
        
        # Sort by hybrid score and get top N
        hybrid_scores.sort(key=lambda x: x[1], reverse=True)
        return [vid for vid, _ in hybrid_scores[:n]]
    
    # If only one approach returned results, use those
    elif als_recs:
        return als_recs
    elif content_recs:
        return content_recs
    
    # If no recommendations could be generated, fall back to popular items
    return popular_recs[:n]


## **5️⃣ Evaluation**
📝 Associated Tasks :
- Choose suitable metrics (e.g., Precision@K, Recall@K, MAP, NDCG).
- Evaluate performance and provide interpretations.

---

For this section we will use several metrics to test our model, these are the metrics and their definitions that can be found in the course or directly with other ressources :
### 1. **Precision@K**

- Measures the **proportion of relevant items** among the top-K recommended items.

$$
\text{Precision@K} = \frac{\text{number of relevant recommended items in top-K}}{K}
$$

- **High Precision@K** → Users are more likely to find something they like within the first K items.  
- **Best for systems prioritizing “hit rate”** (e.g., e-commerce product suggestions).

--- 

### 2. **Mean Reciprocal Rank (MRR)**  
- Measures the **average rank** of the first relevant item across all queries. It’s especially useful for evaluating systems where the goal is to recommend one relevant item at the top.

$$
\text{MRR} = \frac{1}{|Q|} \sum_{q \in Q} \frac{1}{\text{rank}_q}
$$

Where:  
- Q is the set of queries or users.  
- rank_q is the rank of the first relevant item for query \( q \).

**Why use MRR?**  
- Evaluates how quickly the first relevant item appears. A higher MRR means the system performs better in bringing relevant results to the top.

---

### 3. **Hit Rate**  
- Measures the **proportion of users or queries** that have at least one relevant item in the top-K recommendations.

$$
\text{Hit Rate@K} = \frac{\text{number of queries with at least one relevant item in top-K}}{\text{total number of queries}}
$$

**Why use Hit Rate?**  
- Indicates how frequently the system successfully recommends at least one relevant item. A higher Hit Rate means more users are receiving relevant recommendations in the top-K results.
---

### 4. **Normalised Discounted Cumulative Gain (NDCG@K)**  

- Evaluates the quality of the ranking by giving **higher importance to top-ranked relevant items**.

    $$
    \text{DCG@K} = \sum_{i=1}^{K} \frac{rel_i}{\log_2(i + 1)}
    $$

    $$
    \text{NDCG@K} = \frac{\text{DCG@K}}{\text{IDCG@K}}
    $$

Where:  
- \( rel_i \) = relevance of item at position i (binary or graded relevance)  
- \( IDCG@K \) = DCG of an ideal ranking  

**Why use NDCG?**  
- Penalises relevant items that are **ranked lower**.  
- Captures the fact that users are more likely to click on **top-ranked items**.

---

### 5. **Serendipity**  
**Serendipity** extends novelty by focusing on **pleasant surprises**. It suggests items that are both **unexpected** and **relevant**.
$$
\text{Serendipity@K} = \frac{1}{|U|} \sum_{u \in U} \frac{|\text{Serendipitous items}_u|}{K}
$$

Where:
- (Serendipitous items)_u are items that are **relevant to user \( u \)** and **not too similar** to the user’s historical preferences.
- \( |U| \) is the total number of users.

**Example:**  
- If a user loves sci-fi but is recommended an indie time-travel movie that they end up loving, that’s serendipitous.

**Why use Serendipity?**
- Encourages discovery of **diverse content** that users might not have actively searched for.

In [23]:
test_df['user_index'] = test_df['user_id'].map(user_to_index)
test_df['video_index'] = test_df['video_id'].map(video_to_index)

train_user_hist = train_df.groupby("user_id")["video_id"].apply(list).to_dict()
test_user_hist = test_df.groupby("user_id")["video_id"].apply(set).to_dict()

In [27]:

def precision_at_k(recommended_items, relevant_items, k):
    if len(recommended_items) > k:
        recommended_items = recommended_items[:k]
    if not recommended_items:
        return 0.0
    
    hit = len(set(recommended_items) & set(relevant_items))
    return hit / min(k, len(recommended_items))

def mrr_at_k(recommended_items, relevant_items, k):
    recommended_items = recommended_items[:k]
    for i, item in enumerate(recommended_items):
        if item in relevant_items:
            return 1.0 / (i + 1)
    return 0.0

def ndcg_at_k(recommended_items, relevant_items, k):
    if len(recommended_items) > k:
        recommended_items = recommended_items[:k]
    if not recommended_items or not relevant_items:
        return 0.0
    
    # Create a relevance list where 1 if the item is relevant, 0 otherwise
    relevance = [1 if item in relevant_items else 0 for item in recommended_items]
    
    # Calculate DCG
    dcg = 0
    for i, rel in enumerate(relevance):
        # i+1 because we're using 0-based indexing but rank is 1-based
        dcg += rel / np.log2(i + 2)  # log base 2 of rank+1
    
    # Calculate Ideal DCG (IDCG)
    ideal_relevance = [1] * min(len(relevant_items), k)
    idcg = 0
    for i, rel in enumerate(ideal_relevance):
        idcg += rel / np.log2(i + 2)
    
    return dcg / idcg if idcg > 0 else 0

def serendipity_at_k(recommended_items, relevant_items, train_items, popular_items, k):
    if len(recommended_items) > k:
        recommended_items = recommended_items[:k]
    if not recommended_items:
        return 0.0

    serendipitous_hits = 0
    for item in recommended_items:
        if item in relevant_items and item not in train_items and item not in popular_items:
            serendipitous_hits += 1

    return serendipitous_hits / k

def evaluate_hybrid_recommender(train_user_hist, test_user_hist, model, user_item_matrix, 
                               user_to_index, video_to_index, index_to_video,
                               indices, combined_sim, captions, popular_recs,
                               top_k=10, popular_items=None):
    
    if popular_items is None:
        popular_items = set(popular_recs[:top_k])  # Default to top-k most popular videos
    
    hits = 0
    total = 0
    precision_sum = 0
    mrr_sum = 0
    ndcg_sum = 0
    serendipity_sum = 0
    
    for user_id in test_user_hist.keys():
        test_videos = test_user_hist.get(user_id, set())
        train_videos = train_user_hist.get(user_id, [])
        
        if not test_videos:
            continue
        
        seed_video = train_videos[-1] if train_videos else None
        if not seed_video:
            continue
        
        recs = get_hybrid_recommendations(
            user_id, seed_video, model, user_item_matrix,
            user_to_index, video_to_index, index_to_video,
            indices, combined_sim, captions, popular_recs, n=top_k
        )
        
        if not recs:
            continue
        
        if any(video in test_videos for video in recs):
            hits += 1
            
        precision = precision_at_k(recs, test_videos, top_k)
        mrr = mrr_at_k(recs, test_videos, top_k)
        ndcg = ndcg_at_k(recs, test_videos, top_k)
        serendipity = serendipity_at_k(recs, test_videos, train_videos, popular_items, top_k)
        
        precision_sum += precision
        mrr_sum += mrr
        ndcg_sum += ndcg
        serendipity_sum += serendipity
        
        total += 1
    
    print(f"Total users evaluated: {total}")
    
    metrics = {
        f"HR@{top_k}": hits / total if total > 0 else 0,
        f"Precision@{top_k}": precision_sum / total if total > 0 else 0,
        f"MRR@{top_k}": mrr_sum / total if total > 0 else 0,
        f"NDCG@{top_k}": ndcg_sum / total if total > 0 else 0,
        f"Serendipity@{top_k}": serendipity_sum / total if total > 0 else 0
    }
    
    return metrics


In [29]:
top_k = 100
hybrid_metrics = evaluate_hybrid_recommender(
    train_user_hist, test_user_hist, model, user_item_matrix,
    user_to_index, video_to_index, index_to_video,
    indices, combined_sim, captions, popular_recs, top_k
)

print("Hybrid Model Metrics:")
for metric, value in hybrid_metrics.items():
    print(f"{metric}: {value:.4f}")

Total users evaluated: 1411
Hybrid Model Metrics:
HR@100: 1.0000
Precision@100: 0.4274
MRR@100: 0.6665
NDCG@100: 0.4358
Serendipity@100: 0.4265


In [ ]:
len(test_df["user_id"].unique())

1411

### Interpretation and Conclusion

- **Precision@100**  
  - **42.74%** of the videos recommended in the test set are relevant in the top 100 recommendations.  

- **Hit Rate@100**  
  - **100%** of users in the test set received at least one relevant item in their top 10 recommendations.  

- **MRR@100**  
  - The **Mean Reciprocal Rank (MRR)** for the top 10 recommendations is **66.65%** in the test set.  
  - On average, a relevant item appears around the **6th or 7th** position in the top 10 for the training set.  

- **nDCG@100**  
  - The **normalized discounted cumulative gain (nDCG)** at rank 10 is **43.58%** in the test set.  

- **Serendipity@100**  
  - The **Serendipity** score is **42.65%** in the top 100 recommendations.  
  - This suggests that while the model recommends relevant items, it also introduces some diversity and unexpected, yet interesting, recommendations.

## 6️⃣ Trying to Improve Our Model

The following code was used to test different parameter configurations of the **implicit ALS** model in order to find the one yielding the best **nDCG** score.

**nDCG** (Normalized Discounted Cumulative Gain) was chosen as the main evaluation metric because it was the most relevant for our context, for several reasons:

- **nDCG considers the position** of relevant items in the recommendation list.

- Since we recommend a short list of top videos to each user, it's important that the most relevant items appear at the top. nDCG is designed to prioritize this type of relevance.

- In real-world applications, users expect the best results early. If they have to scroll too long without finding relevant content, they may quickly lose interest. nDCG reflects this behavior well by penalizing lower-ranked relevant items.

> **In short**: nDCG provides a position-aware evaluation of the quality of our recommendations.

---

As the grid search took around 30 minutes to run, the corresponding code has been commented out for now.

Below are the best metrics achieved by the model:

> **Best model:**  
> `{'factors': 100, 'regularization': 0.01, 'alpha': 40, 'hit_rate': 0.9978738483345145, 'mrr': 0.6703162791220928, 'ndcg': 0.46914296882680606}`


In [ ]:
"""
best_score = 0
best_params = {}

for factors in [20, 50, 100]:
    for reg in [0.01, 0.1, 0.5]:
        for alpha in [10, 40, 100]:
            print(f"\nTraining ALS with factors={factors}, reg={reg}, alpha={alpha}")
            model = AlternatingLeastSquares(
                factors=factors,
                regularization=reg,
                iterations=15,
                alpha=alpha,
                use_gpu=False
            )
            model.fit(user_item_matrix.T)

            test_recommendations = get_top_n_recommendations(
                model, user_item_matrix, test_users, n=top_n, seen=True
            )

            _, hit_rate, mrr, ndcg = evaluate_recommendations_with_additional_metrics(
                test_recommendations, test_df, top_n=top_n, k=10
            )

            print(f"HitRate@10: {hit_rate:.4f}, MRR@10: {mrr:.4f}, nDCG@10: {ndcg:.4f}")

            if ndcg > best_score:
                best_score = ndcg
                best_params = {
                    'factors': factors,
                    'regularization': reg,
                    'alpha': alpha,
                    'hit_rate': hit_rate,
                    'mrr': mrr,
                    'ndcg': ndcg
                }

print(f"\nBest model: {best_params}")
"""


'\nbest_score = 0\nbest_params = {}\n\nfor factors in [20, 50, 100]:\n    for reg in [0.01, 0.1, 0.5]:\n        for alpha in [10, 40, 100]:\n            print(f"\nTraining ALS with factors={factors}, reg={reg}, alpha={alpha}")\n            model = AlternatingLeastSquares(\n                factors=factors,\n                regularization=reg,\n                iterations=15,\n                alpha=alpha,\n                use_gpu=False\n            )\n            model.fit(user_item_matrix.T)\n\n            test_recommendations = get_top_n_recommendations(\n                model, user_item_matrix, test_users, n=top_n, seen=True\n            )\n\n            _, hit_rate, mrr, ndcg = evaluate_recommendations_with_additional_metrics(\n                test_recommendations, test_df, top_n=top_n, k=10\n            )\n\n            print(f"HitRate@10: {hit_rate:.4f}, MRR@10: {mrr:.4f}, nDCG@10: {ndcg:.4f}")\n\n            if ndcg > best_score:\n                best_score = ndcg\n                

# 7️⃣ Additionnal

You can find :
- The ALS Model which was supposed to be the final notebook [here](models/als.ipynb)
- The implementation of the content-based model without comments (as it was added later) just [here](models/content_based.ipynb).